# Chicago food inspections: why raw counts and normalized rates tell different stories

This notebook represents the explainer notebook for **Project Assignment B**. It details the entire analysis pipeline that was used to create our story on Chicago food inspections, including data cleaning, geospatial joins, and design choices behind our final visuals.

Our analytical approach can be phrased as such:

> Given a time period of **2020-01-01** to **2025-12-31**, which neighborhoods stand out when we account for a comparison between the number of failures per number of inspections rather than just counting the total number of failures in Chicago's food inspections?

**Central thesis.** By accounting for the sheer volume of inspections made during a time period, Chicago's food safety problem shifts focus away from downtown area density and toward a smaller concentration of neighborhoods in **the South and West Sides**.

The difference is the core takeaway from this project. What follows is how the data supports this thesis statement as well as the limits behind it due to changing practices in inspection and reporting.


## 1. Motivation

### What is the dataset?
City of Chicago food inspections database. This database logs date of inspection, results of inspection, facility type, risk rating, address, and coordinates of restaurants and similar facilities. In order to convert the inspections into a neighborhood narrative, I supplement the data with:

- Chicago **community area polygons**, which let me aggregate inspections geographically.
- A **hardship index** table, which adds neighborhood-level socioeconomic context after the spatial join.

### Why choose these datasets?
The topic is perfect for this course since it involves:

- a big real-world administrative dataset,
- geospatial analysis,
- descriptive statistics,
- visualization, and
- a question that is easy to interpret from an urban inequality perspective.

Food inspections are also good for a storytelling assignment because they are directly relatable to readers' experiences. While readers will readily recognize the meaning of a poor inspection result, most of them probably won't consider the distinction between many bad results and a high failure rate.

### Key insight I want the reader to leave with
The project is not just asking **what happened**. It is arguing something sharper:

> **Chicago's inspection problem changes shape once we move from raw counts to normalized and repeated failure measures.**

The dominance of densely populated downtown neighborhoods in the top numbers is due to their high inspection rate. Yet when we factor in the denominator, the narrative will be driven by another geography of recurrent stress. This is what makes the project more than just a leaderboard.

### Goal for the end user's experience
The goal is to provide a non-technical website user experience that takes them through four stages of insight:

1. to see the prevalence of densely populated downtown neighborhoods in **unnormalized counts**, due to their high inspection rate,
2. to realize how **normalization** changes the narrative into the other direction,
3. that **recurrent failures** further validate the shift in the narrative instead of undermining it,
4. to investigate the maps and hovers for a better understanding of locations affected despite normalizing the denominator.

The notebook is technically more advanced than the website, but it is also written with clarity of purpose in mind.


        ## Project Configuration & Environment Setup

        Before diving into data analysis, I first establish a consistent project structure, configure visualization aesthetics, and define reusable utility functions.

        


In [ ]:
from pathlib import Path
import textwrap

import folium
from folium.plugins import HeatMapWithTime
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import Markdown, display


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run the notebook from the repository root or from notebooks/."
    )


PROJECT_ROOT = locate_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

for path in [PROCESSED_DIR, FIGURE_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2020-01-01")
END_DATE = pd.Timestamp("2025-12-31")

RAW_FILES = {
    "inspections": RAW_DIR / "chicago_food_inspections.csv",
    "communities": RAW_DIR / "chicago_community_areas.geojson",
    "hardship": RAW_DIR / "chicago_hardship_index.csv",
}

RESULT_ORDER = ["Pass", "Pass w/ Conditions", "Fail"]

PALETTE = {
    "ink": "#002349",
    "blue": "#002349",
    "coral": "#957C3D",
    "gold": "#C8A64A",
    "teal": "#365C73",
    "sand": "#F7F3EA",
    "slate": "#7D8792",
}

mpl.rcParams.update(
    {
        "figure.facecolor": PALETTE["sand"],
        "axes.facecolor": "white",
        "savefig.facecolor": PALETTE["sand"],
        "axes.edgecolor": PALETTE["ink"],
        "axes.labelcolor": PALETTE["ink"],
        "xtick.color": PALETTE["ink"],
        "ytick.color": PALETTE["ink"],
        "text.color": PALETTE["ink"],
        "axes.titleweight": "bold",
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "font.size": 11.5,
        "font.family": "DejaVu Sans",
        "legend.frameon": False,
    }
)
sns.set_theme(style="whitegrid")


def save_static(fig, filename: str) -> Path:
    output_path = FIGURE_DIR / filename
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    return output_path


def save_plotly(fig, filename: str) -> Path:
    output_path = FIGURE_DIR / filename
    fig.write_html(str(output_path))
    return output_path


def wrap(text: str, width: int = 70) -> str:
    return "\n".join(textwrap.wrap(text, width=width))


def shorten(text: str, width: int = 28) -> str:
    text = str(text)
    if len(text) <= width:
        return text
    return text[: width - 3].rstrip() + "..."


        ## Dataset Catalog Construction & Raw File Validation

        Before officially launching the data cleaning and analysis pipeline, I first construct a clear metadata catalog for the datasets and systematically verify the existence and file size of the local raw files.

        


In [ ]:
dataset_catalog = pd.DataFrame(
    [
        {
            "dataset": "Food inspections",
            "source_url": "https://data.cityofchicago.org/d/4ijn-s7e5",
            "local_file": str(RAW_FILES["inspections"]),
            "role_in_story": "Event-level inspections with outcomes, facility types, and coordinates.",
        },
        {
            "dataset": "Community areas",
            "source_url": "https://data.cityofchicago.org/d/cauq-8yn6",
            "local_file": str(RAW_FILES["communities"]),
            "role_in_story": "Polygon boundaries used to turn points into neighborhood-level measures.",
        },
        {
            "dataset": "Hardship index",
            "source_url": "https://data.cityofchicago.org/d/5kdt-irec",
            "local_file": str(RAW_FILES["hardship"]),
            "role_in_story": "Neighborhood context used to interpret, not explain away, fail-rate patterns.",
        },
    ]
)

file_status = pd.DataFrame(
    [
        {
            "dataset": name,
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1_000_000, 2) if path.exists() else np.nan,
        }
        for name, path in RAW_FILES.items()
    ]
)

assert all(path.exists() for path in RAW_FILES.values()), "Download the three raw data files before running the pipeline."

display(
    Markdown(
        "The project uses one large event table and two contextual datasets. "
        "That structure is a good match for the story because the analysis depends on moving from "
        "**inspection events** to **community areas**, and then adding **neighborhood context**."
    )
)


## 2. Basic stats: let's understand the dataset better

This section does the technical groundwork for the rest of the notebook.

The cleaning and preprocessing choices are:

- keep only the columns that are actually used in the analysis,
- restrict the analysis window to the complete years **2020-2025**,
- keep only the three main outcome labels used in the public story: `Pass`, `Pass w/ Conditions`, and `Fail`,
- remove rows that cannot be placed on the map because they are missing coordinates,
- standardize community area keys before merging,
- perform a spatial join so that inspection points inherit community area labels,
- merge the hardship index only after the geographic assignment is complete.

These choices favor comparability and reproducibility over maximum row retention. In other words, I would rather lose a small number of unusable rows than build figures that mix incompatible outcomes or unmappable inspections.


        ## Data Loading, Cleaning & Spatial Enrichment

        This stage executes data loading, standardized cleaning, spatial joining, and preliminary statistical exploration.
        


In [ ]:
INSPECTION_COLUMNS = [
    "Inspection ID",
    "DBA Name",
    "AKA Name",
    "License #",
    "Facility Type",
    "Risk",
    "Address",
    "Zip",
    "Inspection Date",
    "Inspection Type",
    "Results",
    "Violations",
    "Latitude",
    "Longitude",
]


def load_inspections(path: Path):
    raw_columns = pd.read_csv(path, nrows=0).columns.tolist()
    inspections = pd.read_csv(path, usecols=INSPECTION_COLUMNS, low_memory=False)
    raw_rows = len(inspections)

    inspections["Inspection Date"] = pd.to_datetime(inspections["Inspection Date"], errors="coerce")
    inspections["Latitude"] = pd.to_numeric(inspections["Latitude"], errors="coerce")
    inspections["Longitude"] = pd.to_numeric(inspections["Longitude"], errors="coerce")

    inspections = inspections.loc[inspections["Inspection Date"].between(START_DATE, END_DATE)].copy()
    window_rows = len(inspections)

    inspections["Results"] = inspections["Results"].fillna("").str.strip()
    inspections = inspections.loc[inspections["Results"].isin(RESULT_ORDER)].copy()
    labeled_rows = len(inspections)

    inspections["Risk"] = (
        inspections["Risk"]
        .fillna("Unknown")
        .str.extract(r"(Risk \d)", expand=False)
        .fillna("Unknown")
    )
    inspections["Facility Type"] = inspections["Facility Type"].fillna("Unknown").str.strip()
    inspections["DBA Name"] = inspections["DBA Name"].fillna("Unknown").str.strip()
    inspections["Address"] = inspections["Address"].fillna("Unknown").str.strip()
    inspections["License #"] = inspections["License #"].astype("string")

    missing_coord_rows = int(inspections[["Latitude", "Longitude"]].isna().any(axis=1).sum())
    inspections = inspections.dropna(subset=["Latitude", "Longitude"]).copy()

    inspections["year"] = inspections["Inspection Date"].dt.year
    inspections["month"] = inspections["Inspection Date"].dt.month
    inspections["month_start"] = inspections["Inspection Date"].dt.to_period("M").dt.to_timestamp()
    inspections["is_fail"] = inspections["Results"].eq("Fail").astype(int)
    inspections["is_conditional"] = inspections["Results"].eq("Pass w/ Conditions").astype(int)
    inspections["is_pass"] = inspections["Results"].eq("Pass").astype(int)
    inspections["establishment_key"] = np.where(
        inspections["License #"].notna(),
        inspections["License #"].astype(str),
        inspections["DBA Name"] + " | " + inspections["Address"],
    )

    cleaning_summary = pd.DataFrame(
        [
            {"step": "Rows in imported inspection table", "value": raw_rows},
            {"step": "Rows in 2020-2025 window", "value": window_rows},
            {"step": "Rows with Pass / Pass w/ Conditions / Fail", "value": labeled_rows},
            {"step": "Rows dropped for missing coordinates", "value": missing_coord_rows},
            {"step": "Rows available for mapped analysis", "value": len(inspections)},
            {"step": "Unique establishments", "value": inspections["establishment_key"].nunique()},
            {"step": "Unique facility types", "value": inspections["Facility Type"].nunique()},
        ]
    )

    result_summary = (
        inspections.groupby("Results", as_index=False)
        .agg(inspections=("Inspection ID", "nunique"))
        .assign(share=lambda df: df["inspections"] / df["inspections"].sum() * 100)
        .sort_values("inspections", ascending=False)
    )

    risk_summary = (
        inspections.groupby("Risk", as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
        )
        .assign(fail_rate=lambda df: df["fails"] / df["inspections"] * 100)
        .sort_values("inspections", ascending=False)
    )

    return inspections, raw_columns, cleaning_summary, result_summary, risk_summary


def load_communities(path: Path):
    communities = gpd.read_file(path).rename(
        columns={
            "area_numbe": "community_area_code",
            "community": "community_name",
        }
    )
    communities["community_area_code"] = (
        pd.to_numeric(communities["community_area_code"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )
    communities["community_name"] = communities["community_name"].str.title()
    return communities


def load_hardship(path: Path):
    hardship = pd.read_csv(path).rename(
        columns={
            "Community Area Number": "community_area_code",
            "COMMUNITY AREA NAME": "community_name_hardship",
            "HARDSHIP INDEX": "hardship_index",
        }
    )
    hardship["community_area_code"] = (
        pd.to_numeric(hardship["community_area_code"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )
    hardship["hardship_index"] = pd.to_numeric(hardship["hardship_index"], errors="coerce")
    return hardship


def spatially_enrich(inspections: pd.DataFrame, communities: gpd.GeoDataFrame, hardship: pd.DataFrame):
    inspections_gdf = gpd.GeoDataFrame(
        inspections.copy(),
        geometry=gpd.points_from_xy(inspections["Longitude"], inspections["Latitude"]),
        crs="EPSG:4326",
    )

    inspections_geo = gpd.sjoin(
        inspections_gdf,
        communities[["community_area_code", "community_name", "geometry"]],
        how="left",
        predicate="within",
    ).drop(columns=["index_right"])

    inspections_geo = inspections_geo.merge(
        hardship[["community_area_code", "hardship_index"]],
        on="community_area_code",
        how="left",
    )

    geo_summary = pd.DataFrame(
        [
            {"metric": "Rows after spatial join", "value": len(inspections_geo)},
            {"metric": "Rows missing community area", "value": int(inspections_geo["community_area_code"].isna().sum())},
            {"metric": "Rows with hardship context", "value": int(inspections_geo["hardship_index"].notna().sum())},
            {"metric": "Community areas covered", "value": int(inspections_geo["community_name"].nunique())},
        ]
    )
    return inspections_geo, geo_summary


inspections, raw_columns, cleaning_summary, result_summary, risk_summary = load_inspections(RAW_FILES["inspections"])
communities = load_communities(RAW_FILES["communities"])
hardship = load_hardship(RAW_FILES["hardship"])
inspections_geo, geo_summary = spatially_enrich(inspections, communities, hardship)

basic_stats = pd.DataFrame(
    [
        {"metric": "Date range", "value": f"{inspections_geo['Inspection Date'].min():%Y-%m-%d} to {inspections_geo['Inspection Date'].max():%Y-%m-%d}"},
        {"metric": "Inspection rows used", "value": len(inspections_geo)},
        {"metric": "Unique establishments", "value": inspections_geo["establishment_key"].nunique()},
        {"metric": "Unique facility types", "value": inspections_geo["Facility Type"].nunique()},
        {"metric": "Community areas represented", "value": inspections_geo["community_name"].nunique()},
        {"metric": "Rows missing community area after join", "value": int(inspections_geo["community_area_code"].isna().sum())},
    ]
)

display(
    Markdown(
        f"The cleaned analysis frame contains **{len(inspections_geo):,}** mapped inspections across "
        f"**{inspections_geo['community_name'].nunique()}** community areas. "
        f"Only **{int(inspections_geo['community_area_code'].isna().sum()):,}** rows remain unmatched after the spatial join, "
        "so the neighborhood-level summaries rest on nearly complete geographic coverage."
    )
)


        ## Analysis Table Construction & Results Export

        This stage constructs analytical tables through multi-dimensional aggregation, exports core deliverables, and extracts key insights.
        


In [ ]:
def build_analysis_tables(inspections_geo: gpd.GeoDataFrame, communities: gpd.GeoDataFrame):
    yearly_summary = (
        inspections_geo.groupby("year", as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
            conditional=("is_conditional", "sum"),
            passes=("is_pass", "sum"),
        )
        .sort_values("year")
    )
    yearly_summary["fail_rate"] = yearly_summary["fails"] / yearly_summary["inspections"] * 100
    yearly_summary["conditional_rate"] = yearly_summary["conditional"] / yearly_summary["inspections"] * 100

    community_summary = (
        inspections_geo.groupby(["community_area_code", "community_name"], as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
            conditional=("is_conditional", "sum"),
            passes=("is_pass", "sum"),
            avg_hardship=("hardship_index", "mean"),
        )
    )
    community_summary["fail_rate"] = community_summary["fails"] / community_summary["inspections"] * 100
    community_summary["conditional_rate"] = community_summary["conditional"] / community_summary["inspections"] * 100

    community_compare = community_summary.loc[community_summary["inspections"].ge(300)].copy()
    community_compare["raw_rank"] = community_compare["fails"].rank(method="min", ascending=False)
    community_compare["rate_rank"] = community_compare["fail_rate"].rank(method="min", ascending=False)
    community_compare["rank_shift"] = community_compare["raw_rank"] - community_compare["rate_rank"]

    rank_focus = community_compare.loc[
        community_compare["raw_rank"].le(6)
        | community_compare["rate_rank"].le(6)
        | community_compare["rank_shift"].abs().ge(25)
    ].copy()
    rank_compare = rank_focus.sort_values("raw_rank").copy()

    community_year = (
        inspections_geo.groupby(["community_area_code", "community_name", "year"], as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
        )
    )
    community_year["fail_rate"] = community_year["fails"] / community_year["inspections"] * 100

    repeat_establishments = (
        inspections_geo.groupby(["establishment_key", "community_area_code", "community_name"], as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
        )
    )
    repeat_establishments["repeat_failer"] = repeat_establishments["fails"].ge(2).astype(int)

    repeat_summary = (
        repeat_establishments.groupby(["community_area_code", "community_name"], as_index=False)
        .agg(
            establishments=("establishment_key", "nunique"),
            repeat_failers=("repeat_failer", "sum"),
        )
    )
    repeat_summary["repeat_fail_share"] = repeat_summary["repeat_failers"] / repeat_summary["establishments"] * 100

    repeat_focus = repeat_summary.merge(
        community_summary[["community_area_code", "inspections"]],
        on="community_area_code",
        how="left",
    )
    repeat_focus = repeat_focus.loc[repeat_focus["inspections"].ge(300)].copy()

    facility_counts = inspections_geo["Facility Type"].value_counts()
    top_facility_types = facility_counts.loc[facility_counts.ge(250)].index.tolist()
    facility_summary = (
        inspections_geo.loc[inspections_geo["Facility Type"].isin(top_facility_types)]
        .groupby("Facility Type", as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
            conditional=("is_conditional", "sum"),
        )
    )
    facility_summary["fail_rate"] = facility_summary["fails"] / facility_summary["inspections"] * 100
    facility_summary = facility_summary.sort_values("inspections", ascending=False)

    inspection_type_summary = (
        inspections_geo.groupby("Inspection Type", as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
            conditional=("is_conditional", "sum"),
        )
    )
    inspection_type_summary["fail_rate"] = (
        inspection_type_summary["fails"] / inspection_type_summary["inspections"] * 100
    )
    inspection_type_summary = inspection_type_summary.sort_values("inspections", ascending=False)

    dominant_types = inspection_type_summary.head(6)["Inspection Type"].tolist()
    inspection_type_year = (
        inspections_geo.loc[inspections_geo["Inspection Type"].isin(dominant_types)]
        .groupby(["year", "Inspection Type"], as_index=False)
        .agg(
            inspections=("Inspection ID", "nunique"),
            fails=("is_fail", "sum"),
        )
    )
    inspection_type_year["year_total"] = inspection_type_year["year"].map(
        yearly_summary.set_index("year")["inspections"]
    )
    inspection_type_year["inspection_share"] = (
        inspection_type_year["inspections"] / inspection_type_year["year_total"] * 100
    )
    inspection_type_year["fail_rate"] = (
        inspection_type_year["fails"] / inspection_type_year["inspections"] * 100
    )

    monthly_failed_points = (
        inspections_geo.loc[inspections_geo["is_fail"].eq(1)]
        .assign(
            lat_round=lambda df: df["Latitude"].round(3),
            lon_round=lambda df: df["Longitude"].round(3),
        )
        .groupby(["month_start", "lat_round", "lon_round"], as_index=False)
        .agg(weight=("Inspection ID", "nunique"))
        .sort_values(["month_start", "weight"], ascending=[True, False])
    )

    community_geo = communities.merge(
        community_summary,
        on=["community_area_code", "community_name"],
        how="left",
    ).merge(
        repeat_summary[["community_area_code", "repeat_fail_share"]],
        on="community_area_code",
        how="left",
    )

    context_plot = community_compare.dropna(subset=["avg_hardship"]).copy()
    trend = np.polyfit(context_plot["avg_hardship"], context_plot["fail_rate"], 1)
    context_plot["trend_fit"] = trend[0] * context_plot["avg_hardship"] + trend[1]
    context_plot["residual"] = context_plot["fail_rate"] - context_plot["trend_fit"]

    return {
        "yearly_summary": yearly_summary,
        "community_summary": community_summary,
        "community_compare": community_compare,
        "rank_compare": rank_compare,
        "community_year": community_year,
        "repeat_summary": repeat_summary,
        "repeat_focus": repeat_focus,
        "facility_summary": facility_summary,
        "inspection_type_summary": inspection_type_summary,
        "inspection_type_year": inspection_type_year,
        "monthly_failed_points": monthly_failed_points,
        "community_geo": community_geo,
        "context_plot": context_plot,
        "hardship_trend": trend,
    }


analysis_tables = build_analysis_tables(inspections_geo, communities)

tables_to_export = {
    "yearly_summary": analysis_tables["yearly_summary"],
    "community_summary": analysis_tables["community_summary"],
    "community_compare": analysis_tables["community_compare"],
    "community_year": analysis_tables["community_year"],
    "rank_compare": analysis_tables["rank_compare"],
    "facility_summary": analysis_tables["facility_summary"],
    "inspection_type_summary": analysis_tables["inspection_type_summary"],
    "inspection_type_year": analysis_tables["inspection_type_year"],
    "repeat_summary": analysis_tables["repeat_summary"],
    "repeat_focus": analysis_tables["repeat_focus"],
    "monthly_failed_points": analysis_tables["monthly_failed_points"],
}

for name, df in tables_to_export.items():
    df.to_csv(TABLE_DIR / f"{name}.csv", index=False)

table_inventory = pd.DataFrame(
    [{"table": name, "rows": len(df)} for name, df in tables_to_export.items()]
)

community_compare = analysis_tables["community_compare"]
context_plot = analysis_tables["context_plot"]
inspection_type_summary = analysis_tables["inspection_type_summary"]

top_raw = community_compare.nlargest(1, "fails").iloc[0]
top_rate = community_compare.nlargest(1, "fail_rate").iloc[0]
biggest_drop = community_compare.nsmallest(1, "rank_shift").iloc[0]
biggest_rise = community_compare.nlargest(1, "rank_shift").iloc[0]
hardship_corr = context_plot["avg_hardship"].corr(context_plot["fail_rate"])
complaint_fail_rate = float(
    inspection_type_summary.loc[
        inspection_type_summary["Inspection Type"].eq("Complaint"),
        "fail_rate",
    ].iloc[0]
)

display(
    Markdown(
        f"**Thesis checkpoint.** Across communities with at least **300 inspections**, the largest raw-total leader is "
        f"**{top_raw['community_name']}** with **{int(top_raw['fails'])}** failed inspections, "
        f"while the highest fail-rate community is **{top_rate['community_name']}** at "
        f"**{top_rate['fail_rate']:.1f} failures per 100 inspections**. "
        f"The strongest raw-to-rate drop is **{biggest_drop['community_name']}**, which falls from rank "
        f"**{int(biggest_drop['raw_rank'])}** to **{int(biggest_drop['rate_rank'])}**, while "
        f"**{biggest_rise['community_name']}** rises from **{int(biggest_rise['raw_rank'])}** to "
        f"**{int(biggest_rise['rate_rank'])}**. "
        f"At the same time, inspection composition matters: **Complaint** inspections fail at "
        f"**{complaint_fail_rate:.1f}%**, so citywide trend lines also reflect what kinds of inspections were performed. "
        f"The hardship-to-fail-rate relationship remains positive but partial (`r = {hardship_corr:.2f}`), "
        f"which supports using hardship as context rather than as a full explanation."
    )
)


        ## Visualization Chart Construction

        This stage constructs multi-dimensional visualizations using Matplotlib, Plotly, and Folium to intuitively present analytical findings.
        


In [ ]:
def plot_citywide_trend(yearly_summary: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.4))

    axes[0].bar(yearly_summary["year"], yearly_summary["inspections"], color=PALETTE["blue"], width=0.7)
    axes[0].set_title("Inspection volume climbs after 2021")
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Inspection count")
    axes[0].set_xticks(yearly_summary["year"])

    ax0b = axes[0].twinx()
    ax0b.plot(yearly_summary["year"], yearly_summary["fails"], color=PALETTE["coral"], marker="o", linewidth=2.5)
    ax0b.set_ylabel("Failed inspections")
    for row in yearly_summary.itertuples():
        ax0b.text(row.year, row.fails + 35, f"{int(row.fails)}", ha="center", va="bottom", fontsize=9, color=PALETTE["coral"])

    axes[1].plot(yearly_summary["year"], yearly_summary["fail_rate"], color=PALETTE["coral"], marker="o", linewidth=2.5)
    axes[1].plot(yearly_summary["year"], yearly_summary["conditional_rate"], color=PALETTE["slate"], marker="o", linewidth=2.3)
    axes[1].set_title("Rates move in a tighter band than raw counts")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Rate per 100 inspections")
    axes[1].set_xticks(yearly_summary["year"])
    axes[1].legend(["Fail rate", "Pass w/ Conditions rate"], loc="upper right")

    fig.suptitle(
        wrap("Figure 1. Chicago inspected more places after 2021, but the failure rate stayed in a much narrower range", 84),
        fontsize=18,
        color=PALETTE["ink"],
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
    return fig


def plot_raw_vs_rate(community_compare: pd.DataFrame):
    top_raw = community_compare.nlargest(10, "fails").sort_values("fails")
    top_rate = community_compare.nlargest(10, "fail_rate").sort_values("fail_rate")

    fig, axes = plt.subplots(1, 2, figsize=(14.6, 7.8))

    axes[0].barh(top_raw["community_name"], top_raw["fails"], color=PALETTE["blue"])
    axes[0].set_title("Raw fail totals mostly track inspection density")
    axes[0].set_xlabel("Failed inspections, 2020-2025")
    for value, y in zip(top_raw["fails"], range(len(top_raw))):
        axes[0].text(value + 15, y, f"{int(value)}", va="center", fontsize=9)

    axes[1].barh(top_rate["community_name"], top_rate["fail_rate"], color=PALETTE["coral"])
    axes[1].set_title("Fail rate reveals a different set of communities")
    axes[1].set_xlabel("Fails per 100 inspections, 2020-2025")
    for value, y in zip(top_rate["fail_rate"], range(len(top_rate))):
        axes[1].text(value + 0.35, y, f"{value:.1f}", va="center", fontsize=9)

    fig.suptitle(
        wrap("Figure 2. Downtown dominates raw counts, but normalized failure rates shift attention toward South and West Side communities", 86),
        fontsize=18,
        color=PALETTE["ink"],
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=3.4)
    return fig


def plot_repeat_fail_share(repeat_focus: pd.DataFrame):
    plot_data = (
        repeat_focus.nlargest(10, "repeat_fail_share")
        .sort_values("repeat_fail_share")
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(10.8, 6.8))
    ax.barh(plot_data["community_name"], plot_data["repeat_fail_share"], color=PALETTE["teal"])
    ax.set_title("Figure 3. Some communities have a much larger share of repeat-failing establishments")
    ax.set_xlabel("Repeat-failing establishments per 100 establishments")

    for y, value in enumerate(plot_data["repeat_fail_share"]):
        ax.text(value + 0.6, y, f"{value:.1f}", va="center", fontsize=9)

    return fig


def plot_hardship_context(context_plot: pd.DataFrame, trend: np.ndarray):
    label_points = pd.concat(
        [
            context_plot.nlargest(3, "residual"),
            context_plot.nsmallest(3, "residual"),
        ]
    ).drop_duplicates(subset=["community_name"])

    fig, ax = plt.subplots(figsize=(11, 7.5))
    ax.scatter(
        context_plot["avg_hardship"],
        context_plot["fail_rate"],
        s=np.sqrt(context_plot["inspections"]) * 20,
        color=PALETTE["blue"],
        edgecolor=PALETTE["ink"],
        linewidth=0.8,  
        alpha=0.85,
        zorder=3
    )

    x_line = np.linspace(context_plot["avg_hardship"].min(), context_plot["avg_hardship"].max(), 200)
    y_line = trend[0] * x_line + trend[1]
    ax.plot(x_line, y_line, color=PALETTE["coral"], linewidth=2.6, zorder=2)

    for row in label_points.itertuples():
        ax.text(row.avg_hardship + 1.8, row.fail_rate + 0.5, row.community_name, fontsize=10, fontweight="medium", color=PALETTE["ink"], zorder=4 )

    ax.set_title("Figure 4. Higher-hardship communities tend to have higher fail rates, with clear exceptions")
    ax.set_xlabel("Hardship index")
    ax.set_ylabel("Fails per 100 inspections")
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=PALETTE["blue"], edgecolor=PALETTE["ink"], label="Community (size = inspection count)"),
        plt.Line2D([0], [0], color=PALETTE["coral"], lw=3, label="Trend line")
    ]
    ax.legend(handles=legend_elements, loc="upper left", frameon=False, fontsize=10)
    fig.tight_layout()
    return fig


def plot_inspection_type_mix(inspection_type_year: pd.DataFrame, inspection_type_summary: pd.DataFrame):
    share_pivot = (
        inspection_type_year.pivot(index="year", columns="Inspection Type", values="inspection_share")
        .fillna(0)
        .sort_index()
    )
    share_order = inspection_type_year.groupby("Inspection Type")["inspections"].sum().sort_values(ascending=False).index
    share_pivot = share_pivot.reindex(columns=share_order)

    type_rate_plot = (
        inspection_type_summary.loc[inspection_type_summary["inspections"].ge(500)]
        .sort_values("fail_rate")
        .copy()
    )
    type_rate_plot["type_label"] = type_rate_plot["Inspection Type"].map(lambda x: shorten(x, 28))

    fig, axes = plt.subplots(1, 2, figsize=(15.2, 7.0))

    share_colors = ["#002349", "#365C73", "#957C3D", "#C8A64A", "#7D8792", "#B7A98A"]
    share_pivot.plot(
        kind="bar",
        stacked=True,
        ax=axes[0],
        color=share_colors[: len(share_pivot.columns)],
        width=0.72,
    )
    axes[0].set_title("The inspection mix shifts modestly across years")
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Share of all inspections (%)")
    axes[0].legend(title="Inspection type", fontsize=8.5, title_fontsize=9, loc="lower center", bbox_to_anchor=(0.5, 1.12), ncol=3, borderaxespad=0, frameon=False)
    axes[0].tick_params(axis='x', rotation=0)
    
    axes[1].barh(type_rate_plot["type_label"], type_rate_plot["fail_rate"], color=PALETTE["gold"])
    axes[1].set_title("Those inspection types have very different fail rates")
    axes[1].set_xlabel("Fails per 100 inspections")
    for value, y in zip(type_rate_plot["fail_rate"], range(len(type_rate_plot))):
        axes[1].text(value + 0.35, y, f"{value:.1f}", va="center", fontsize=9)

    fig.suptitle(
        wrap("Figure 6. Trend comparisons also depend on what kinds of inspections the city performed", 86),
        fontsize=18,
        color=PALETTE["ink"],
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.88], w_pad=2.8)
    return fig


def build_heat_map(monthly_failed_points: pd.DataFrame):
    month_index = pd.period_range("2020-01", "2025-12", freq="M").to_timestamp()
    heat_frames = []
    heat_labels = []

    for month_start in month_index:
        month_data = monthly_failed_points.loc[monthly_failed_points["month_start"].eq(month_start)].nlargest(400, "weight")
        frame = month_data[["lat_round", "lon_round", "weight"]].values.tolist()
        heat_frames.append(frame)
        heat_labels.append(month_start.strftime("%Y-%m"))

    heat_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10, tiles="CartoDB Positron")
    HeatMapWithTime(
        data=heat_frames,
        index=heat_labels,
        auto_play=False,
        radius=18,
        max_opacity=0.82,
        use_local_extrema=False,
        gradient={0.2: "#7D8792", 0.55: "#957C3D", 1.0: "#002349"},
    ).add_to(heat_map)
    return heat_map


def build_choropleth(community_geo: gpd.GeoDataFrame):
    choropleth_ready = community_geo.copy()
    choropleth_ready["fail_rate"] = choropleth_ready["fail_rate"].fillna(0)
    choropleth_ready["repeat_fail_share"] = choropleth_ready["repeat_fail_share"].fillna(0)
    choropleth_ready["inspections"] = choropleth_ready["inspections"].fillna(0)
    choropleth_ready["fails"] = choropleth_ready["fails"].fillna(0)
    choropleth_ready["avg_hardship"] = choropleth_ready["avg_hardship"].fillna(np.nan)

    fig = px.choropleth(
        choropleth_ready,
        geojson=choropleth_ready.__geo_interface__,
        locations=choropleth_ready.index,
        color="fail_rate",
        color_continuous_scale=["#F7F3EA", "#957C3D", "#002349"],
        hover_name="community_name",
        hover_data={
            "fail_rate": ":.2f",
            "fails": True,
            "inspections": True,
            "repeat_fail_share": ":.2f",
            "avg_hardship": ":.1f",
        },
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(
        title="Figure 5. Community fail rate",
        margin=dict(l=10, r=10, t=60, b=10),
        paper_bgcolor=PALETTE["sand"],
        font=dict(color=PALETTE["ink"]),
    )
    return fig


## 3. Data analysis

This section contains the main analysis behind the project narrative.

The argument builds in layers:

- start with the citywide denominator,
- compare raw community totals with normalized community rates,
- test whether repeat failures are concentrated,
- inspect the hotspot geography over time,
- then place the result in a broader neighborhood context.

The goal is not to produce a single "winner" or "worst neighborhood" ranking. Instead, the goal is to show how the interpretation changes when the denominator and the geography are handled carefully.


        ## Figure 1. More inspections did not automatically mean a worse failure picture

        The first chart does two jobs.

        - The left panel keeps the inspection denominator visible.
        - The right panel shows the rates that matter once total volume changes.

        This follows the basic charting rule that the denominator should stay visible when rates are the real message.

In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)

yearly_summary = analysis_tables["yearly_summary"]

fig1 = plot_citywide_trend(yearly_summary)
save_static(fig1, "figure1_citywide_food_safety_trend_2020_2025.png")
plt.show()

**Caption.** Inspection volume rises after 2021, but the failure rate remains much steadier than the raw failure count. This is the first warning that raw totals alone are not enough for comparing neighborhoods.


        ## Figure 2. Raw fail counts and fail rate point to different parts of the city

        This chart is where the story becomes spatially interesting.

        The left panel rewards size. The right panel rewards intensity. To keep tiny denominators from dominating the rate ranking, I compare only community areas with at least `300` inspections over the full period.
        

In [ ]:
fig2 = plot_raw_vs_rate(community_compare)
save_static(fig2, "figure2_raw_vs_rate_community_2020_2025.png")
plt.show()


**Caption.** Near North Side, the Loop, and Near West Side are major raw-count leaders, but they do not remain at the top when failures are divided by total inspections. The normalized ranking pulls attention toward South and West Side communities that would otherwise be overshadowed by downtown inspection volume.


        ## Figure 3. Some communities have a much larger share of repeat-failing establishments

        This chart does one primary job

        - It ranks and displays the 10 communities with the highest share of repeatedly failing food establishments, making stark geographic disparities in persistent food safety issues immediately visible.

        This follows the analytical principle that focusing on repeat violations reveals communities with sustained, unresolved food safety challenges, rather than just one-time inspection failures.
        

In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)
globals().update(analysis_tables)
repeat_focus = analysis_tables["repeat_focus"]
fig3 = plot_repeat_fail_share(repeat_focus)
save_static(fig3, "figure3_repeat_fail_share_community_2020_2025.png")
plt.show()


**Caption.** This figure quantifies recurrence rather than one-off failure events. Communities such as East Side, Auburn Gresham, and South Chicago do not only rank highly on fail rate; they also have a larger share of establishments that fail repeatedly, which strengthens the interpretation that some local problem patterns persist over time.


        ## Figure 4. Neighborhood hardship matters, but it is not the whole story

        This is a context chart, not a causal claim.

        I use it to ask whether the rate pattern has a broad social gradient and which communities sit meaningfully above or below that line.
        


In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)
globals().update(analysis_tables)
hardship_trend = analysis_tables["hardship_trend"]
fig4 = plot_hardship_context(context_plot, hardship_trend)
save_static(fig4, "figure4_hardship_vs_fail_rate_2020_2025.png")
plt.show()


**Caption.** Higher-hardship communities tend to have higher fail rates, but the relationship is not deterministic. Several communities sit well above or below the fitted line, which is exactly why hardship should be treated as context rather than a sufficient explanation.


        ## Figure 5. The neighborhood map looks different once I divide by inspection volume

        The heatmap shows repeated hotspots at the point level. This choropleth answers the neighborhood question directly: which community areas stay weak once failures are normalized by how many inspections happened there?
        


In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)
globals().update(analysis_tables)
community_geo = analysis_tables["community_geo"]
fig5 = build_choropleth(community_geo)
save_plotly(fig5, "figure5_community_fail_rate_map_2020_2025.html")
fig5.show()


**Caption.** The choropleth is the clearest neighborhood-level summary of the project. It makes the denominator visible while preserving geographic context, so the reader can compare communities without confusing restaurant density for inspection weakness.


        ## Figure 6. Trend comparisons also depend on what kinds of inspections the city performed

        This chart does two jobs.

        - The left panel illustrates the annual composition of inspection types, showing modest shifts in the mix of inspections carried out across years.
        - The right panel compares failure rates across different inspection types, revealing large disparities in failure intensity between inspection categories.

        This follows the critical principle that accurate trend analysis must account for changes in inspection types, not just total inspection volume.
        


In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)
globals().update(analysis_tables)
inspection_type_year = analysis_tables["inspection_type_year"]
inspection_type_summary = analysis_tables["inspection_type_summary"]
fig6 = plot_inspection_type_mix(inspection_type_year, inspection_type_summary)
save_static(fig6, "figure6_inspection_type_mix_2020_2025.png")
plt.show()


**Caption.** This figure is part analysis and part critical reflection. Complaint inspections fail far more often than routine canvass or re-inspection visits, and the share of those inspection types shifts over time. That means annual changes in overall fail rate should not be read as pure changes in food-safety conditions; they also reflect what the city chose or needed to inspect.


        ## Figure 7. `HeatMapWithTime`: failed inspections keep returning to a limited set of places

        This is the temporal map view.

        Each frame is one month. I keep the strongest weighted cells in each month so the animation stays readable instead of turning into one blurred blanket over the whole city.
        


In [ ]:
analysis_tables = build_analysis_tables(inspections_geo, communities)
monthly_failed_points = analysis_tables["monthly_failed_points"]

heat_map = build_heat_map(monthly_failed_points)
heat_map.save(str(FIGURE_DIR / "figure7_failed_inspections_timeheatmap_2020_2025.html"))
heat_map

**Caption.** The monthly heatmap shows that failed inspections do not spread evenly across the city. They keep returning to a more limited set of corridors, which supports the claim that citywide averages hide recurring local hotspots.


## 4. Genre

The intended public-facing project is best described as an **annotated-chart story with partitioned-poster sections**, supported by interactive maps. That is a good fit because the narrative has one main analytical twist, but the reader still benefits from being able to inspect neighborhoods on their own.

### Visual narrative strategies
Like Segel & Heer's examples, the notebook and website utilize all three categories of visual narrative strategies:

- **Visual structuring**: consistent numberings of figures, static color choices, repeatable units ("fails per 100 inspections"), and transition from city-wide to community to point-level maps.
- **Highlighting**: distinctive color scheme (coral) for failure-related metrics, explanations of takeaways through captions, and naming particular communities in slope and hardship graphs.
- **Guiding the transition between states of engagement**: transitioning from denominator-aware summary statistics to interactive maps in the right sequence, so that readers are not thrown into the map before understanding its reasoning.

### Narrative structure tactics
The work also incorporates all three categories of narrative structure tactics proposed by Segel & Heer:

- **Ordering**: the notebook mostly follows a linear storyline, since the analysis is predicated on the reader knowing about the denominator problem and the data processing pipeline.
- **Interactivity**: website-level maps provide hover, panning, zooming, and timeline controls that enable discovery without taking the user away from the storyline.
- **Explicit messaging**: explicit captions, text sections, and comparison context explain what question each graph addresses.

This provides the necessary narrative structure to the project that guides the audience, but does not overly constrain them.


## 5. Visualizations

The final website uses seven visualizations in the same order as the notebook:

- **Figure 1: citywide bars and lines**, to keep inspection volume visible before discussing rates.
- **Figure 2: raw counts versus normalized rates**, to show why the denominator changes the story.
- **Figure 3: repeat-failure bar chart**, to separate persistent failure from one-off inspection outcomes.
- **Figure 4: hardship scatter plot with fitted trend line**, to add community context without making a causal claim.
- **Figure 5: interactive choropleth map**, to turn the rate comparison into a neighborhood geography.
- **Figure 6: inspection composition chart**, to show why temporal comparisons need caution.
- **Figure 7: HeatMapWithTime**, to show spatial repetition through monthly frames.

This visualization set supports a magazine-style story: first establish the denominator issue, then show the communities that remain high after normalization, then add context and time.


## 6. Discussion

### Restating the main insight
While the most powerful point might be that certain neighborhoods fail inspections at greater rates than others, the truth is actually that the nature of the problem changes completely once **inspections are considered together with repeat inspection failures**. Downtown neighborhoods top the list of simple numbers, but it is a much shorter list of South and West Side neighborhoods that appear problematic.

### Data limitations
The notebook also has important limits:

- The results of inspections are influenced by both the actions of the establishments and the practices of city inspectors; therefore, they are not a definitive indicator of food safety in the neighborhoods.
- Not all establishments are equally inspected, which increases the chances of both mistakes and improvements.
- The hardship index is rather stable and rough in comparison to the inspection schedule and can be considered contextual data but not an explanatory factor.
- There are only a few missing observations, and the community-level measures can still be vulnerable to the denominator effect despite using the 300-inspections rule.

### Changes in reporting and inspection practice over time
This is the most important caution for the temporal analysis. Inspection types do not have the same failure profile:

- Complaint inspections have a tendency to fail far more often than regular canvass inspections;
- Re-inspections have a tendency to fail far less often than first-time complaint or license inspections; and
- These proportions vary on a year-to-year basis.

So even when the overall fail rate looks stable, part of that stability or movement may come from a changing inspection mix rather than a pure change in underlying food-safety conditions. The new inspection-type figure makes that limitation visible in the analysis instead of leaving it implicit.

### Why might these patterns occur?
The notebook can suggest plausible mechanisms, but it cannot prove them causally. The observed geography could reflect several overlapping forces:

- variation in the composition of establishments between neighborhoods,
- uneven distributions of repeat offenders within localities,
- complaint-oriented enforcement in certain locations,
- variation in business turnover and profit margins, and
- general conditions in neighborhoods that reflect hardship without necessarily being caused by hardship.

In other words, the high failure rates could be attributed to poorer compliance, better targeting, a difference in establishments, or possibly some combination of the three. This is why the best value in this project lies in its descriptive and spatial reasoning rather than causation.

### What would make the project even stronger?
Longer-term modeling at an establishment level could prove to be a useful extension, such as modeling recurrent events across time by establishment types within communities. An additional useful extension would be modeling by inspection types since this would control for changes in outcomes due to differences in inspections.


## 7. Contributions

| Group member | Main contributions |
|---|---|
| Siyu Xia | Dataset search and selection; initial notebook structure; figure optimization; website design and deployment |
| Xuanyuan Cao | Notebook construction; data cleaning; data analysis |
| Chen Hou | Data analysis; notebook optimization; figure optimization |


## 8. References

- City of Chicago. **Food Inspections**. https://data.cityofchicago.org/d/4ijn-s7e5
- City of Chicago. **Boundaries - Community Areas (current)**. https://data.cityofchicago.org/d/cauq-8yn6
- City of Chicago. **Hardship Index**. https://data.cityofchicago.org/d/5kdt-irec
- Segel, Edward, and Jeffrey Heer. **Narrative Visualization: Telling Stories with Data**. *IEEE Transactions on Visualization and Computer Graphics*, 16(6), 2010. https://idl.uw.edu/papers/narrative
- Course wiki. **Final Project - Project Assignment B**. https://github.com/suneman/socialdata2026/wiki/Final-Project
